# HybridLLMRouter - Training

This notebook demonstrates how to train the **HybridLLMRouter**.

## Overview

HybridLLMRouter routes between small and large models using an MLP regressor.
It supports different routing modes: deterministic, probabilistic, and transformed.

**Key Features**:
- Binary routing (small vs large model)
- MLP-based decision making
- Configurable routing threshold

## 1. Environment Setup

In [1]:
# Install required packages (for Colab)
!git clone https://github.com/ulab-uiuc/LLMRouter.git
%cd LLMRouter
!pip install -e .
!pip install transformers torch


Cloning into 'LLMRouter'...
remote: Enumerating objects: 6017, done.
remote: Counting objects: 100% (172/172), done.
remote: Compressing objects: 100% (91/91), done.
remote: Total 6017 (delta 98), reused 87 (delta 81), pack-reused 5845 (from 2)
Receiving objects: 100% (6017/6017), 89.41 MiB | 53.05 MiB/s, done.
Resolving deltas: 100% (2946/2946), done.
Updating files: 100% (288/288), done.
/home/zhongjie/LLMRouter
Obtaining file:///home/zhongjie/LLMRouter
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for llmrouter-lib (pyproject.toml) ... done
  Created wheel for llmrouter-lib: filename=llmrouter_lib-0.2.0-0.editable-py3-none-any.whl size=15709 sha256=ba03c8a03a8d719114455fce29065dff7e24993dcf454c700113246d50a7b242
  Stored in directory: /tmp/pip-ephem-wheel-cache-dox4tdl9/wheels/82/4a/fd/59c4aec93c356c380d

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'your-key'
os.environ['ANTHROPIC_API_KEY'] = 'your-key'
# Or for multiple keys:
os.environ['API_KEYS'] = '["key1", "key2"]'

In [3]:
from llmrouter.models.hybrid_llm import HybridLLMRouter, HybridLLMTrainer
from llmrouter.utils import setup_environment

setup_environment()
print("Environment setup complete!")

Environment setup complete!


## 2. Configuration

HybridLLMRouter uses the following configuration parameters:

| Parameter | Description | Default |
|-----------|-------------|--------|
| `router_mode` | Routing mode | "probabilistic" |
| `router_tau` | Temperature for probabilistic routing | 0.1 |
| `router_threshold` | Decision threshold | 0.5 |
| `hidden_layer_sizes` | MLP architecture | [128, 64] |

In [4]:
import yaml

CONFIG_PATH = "configs/model_config_train/hybrid_llm.yaml"

with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

print("Current Configuration:")
print("=" * 50)
print(yaml.dump(config, default_flow_style=False))

Current Configuration:
data_path:
  llm_data: data/example_data/llm_candidates/default_llm.json
  llm_embedding_data: data/example_data/llm_candidates/default_llm_embeddings.json
  query_data_test: data/example_data/query_data/default_query_test.jsonl
  query_data_train: data/example_data/query_data/default_query_train.jsonl
  query_embedding_data: data/example_data/routing_data/query_embeddings_longformer.pt
  routing_data_test: data/example_data/routing_data/default_routing_test_data.jsonl
  routing_data_train: data/example_data/routing_data/default_routing_train_data.jsonl
hparam:
  activation: relu
  hidden_layer_sizes:
  - 128
  - 64
  max_iter: 300
  solver: adam
model_path:
  ini_model_path: configs/hybrid_ini.pkl
  load_model_path: configs/hybrid_trained.pkl
  save_model_path: configs/hybrid_trained.pkl
router_mode: probabilistic
router_tau: 0.1
router_threshold: 0.5



## 3. Initialize Router

In [5]:
router = HybridLLMRouter(yaml_path=CONFIG_PATH)

print("Router initialized successfully!")
print(f"Router mode: {config.get('router_mode', 'probabilistic')}")
print(f"Threshold: {config.get('router_threshold', 0.5)}")

✅ MetaRouter initialized successfully (YAML + data loaded).
[HybridLLMRouter] Filtered to 4 models with routing data.
[HybridLLMRouter] Mode=probabilistic, Small='qwen2.5-7b-instruct', Large='llama-3.3-nemotron-super-49b-v1'
Router initialized successfully!
Router mode: probabilistic
Threshold: 0.5


## 4. Training

In [6]:
trainer = HybridLLMTrainer(router=router, device='cpu')

print("Trainer initialized!")

[HybridLLMTrainer] Initialized. Mode=probabilistic
Trainer initialized!


In [7]:
print("Starting training...")
print("=" * 50)

trainer.train()

print("=" * 50)
print("Training completed!")

Starting training...
[HybridLLMTrainer] Training on 5616 samples... mode=probabilistic
[HybridLLMTrainer] Saving final model: /home/zhongjie/LLMRouter/llmrouter/configs/hybrid_trained.pkl
Created directory: /home/zhongjie/LLMRouter/llmrouter/configs
Successfully saved pickle model: /home/zhongjie/LLMRouter/llmrouter/configs/hybrid_trained.pkl
Training completed!


## 5. Model Verification

In [8]:
# Test prediction
test_query = {"query": "What is the capital of France?"}
result = router.route_single(test_query)

print(f"Test query: {test_query['query']}")
print(f"Routed to: {result['model_name']}")

Successfully loaded pickle model: /home/zhongjie/LLMRouter/llmrouter/configs/hybrid_trained.pkl


Input ids are automatically padded to be a multiple of `config.attention_window`: 512


Test query: What is the capital of France?
Routed to: qwen2.5-7b-instruct


## Summary

In this notebook, we:

1. **Loaded Configuration**: Set up HybridLLMRouter with YAML configuration
2. **Trained Model**: Learned routing decision boundary
3. **Verified Model**: Tested routing with sample queries

**Routing Modes**:
- **deterministic**: Hard threshold decision
- **probabilistic**: Soft probability-based routing
- **transformed**: Temperature-scaled probabilities

**Next Steps**:
- Use the next part of notebook for inference

# HybridLLMRouter - Inference

This notebook demonstrates how to use a trained **HybridLLMRouter** for inference.

## 1. Environment Setup (optional)

In [ ]:
from llmrouter.models.hybrid_llm import HybridLLMRouter
from llmrouter.utils import setup_environment
import yaml

setup_environment()

## 2. Load Trained Router

In [9]:
CONFIG_PATH = "configs/model_config_train/hybrid_llm.yaml"

router = HybridLLMRouter(yaml_path=CONFIG_PATH)
print("Router loaded!")

✅ MetaRouter initialized successfully (YAML + data loaded).
[HybridLLMRouter] Filtered to 4 models with routing data.
[HybridLLMRouter] Mode=probabilistic, Small='qwen2.5-7b-instruct', Large='llama-3.3-nemotron-super-49b-v1'
Router loaded!


## 3. Query Routing

In [10]:
EXAMPLE_QUERIES = [
    {"query": "What is 2 + 2?"},
    {"query": "Explain quantum entanglement."},
    {"query": "Write a sorting algorithm."},
]

print("Routing Results:")
print("=" * 60)

for i, query in enumerate(EXAMPLE_QUERIES, 1):
    result = router.route_single(query)
    print(f"{i}. {query['query'][:50]}...")
    print(f"   Routed to: {result['model_name']}")

Routing Results:
Successfully loaded pickle model: /home/zhongjie/LLMRouter/llmrouter/configs/hybrid_trained.pkl
1. What is 2 + 2?...
   Routed to: qwen2.5-7b-instruct
Successfully loaded pickle model: /home/zhongjie/LLMRouter/llmrouter/configs/hybrid_trained.pkl
2. Explain quantum entanglement....
   Routed to: qwen2.5-7b-instruct
Successfully loaded pickle model: /home/zhongjie/LLMRouter/llmrouter/configs/hybrid_trained.pkl
3. Write a sorting algorithm....
   Routed to: llama-3.3-nemotron-super-49b-v1


## 4. File-Based Inference

Load queries from a file and save results.

In [11]:
import json

# Load queries from a JSONL file
def load_queries_from_file(file_path):
    """Load queries from a JSONL file."""
    queries = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                queries.append(json.loads(line))
    return queries

# Save results to a JSONL file
def save_results_to_file(results, output_path):
    """Save routing results to a JSONL file."""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        for result in results:
            f.write(json.dumps(result, ensure_ascii=False) + '\n')
    print(f"Results saved to: {output_path}")

# Example: Load from default query file
QUERY_FILE = "data/example_data/query_data/default_query_test.jsonl"
OUTPUT_FILE = "outputs/hybrid_llm_router_results.jsonl"

if os.path.exists(QUERY_FILE):
    # Load queries
    file_queries = load_queries_from_file(QUERY_FILE)
    print(f"Loaded {len(file_queries)} queries from: {QUERY_FILE}")
    
    # Route queries
    file_results = router.route_batch(batch=file_queries[:10])
    print(f"Routed {len(file_results)} queries")
    
    # Save results
    save_results_to_file(file_results, OUTPUT_FILE)
    
    # Show sample results
    print(f"\nSample results:")
    for i, result in enumerate(file_results[:3], 1):
        print(f"  {i}. {result.get('query', '')[:40]}... -> {result['model_name']}")
else:
    print(f"Query file not found: {QUERY_FILE}")
    print("Create a JSONL file with format: {\"query\": \"Your question\"}")

Loaded 706 queries from: data/example_data/query_data/default_query_test.jsonl
Successfully loaded pickle model: /home/zhongjie/LLMRouter/llmrouter/configs/hybrid_trained.pkl

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

Routed 10 queries
Results saved to: outputs/hybrid_llm_router_results.jsonl

Sample results:
  1. Q: There are 4 houses in a row, numbered... -> qwen2.5-7b-instruct
  2. Q: There are 3 houses in a row, numbered... -> qwen2.5-7b-instruct
  3. Q: There are 3 houses in a row, numbered... -> llama-3.3-nemotron-super-49b-v1


## Summary

HybridLLMRouter provides:
- Efficient binary routing between small and large models
- Configurable routing modes for different use cases